# 2. Fine-tune Qwen3.5-4B for the folder organizer

Runtime: **Runtime > Change runtime type > T4 GPU** (free).

- Trains a LoRA adapter (about 1% of the weights) in 16-bit. Unsloth advises against 4-bit QLoRA for Qwen3.5.
- Each example is cut exactly where the app asks the model, and only the model's own turn is trained.
- At the end it saves the adapter and a `q4_k_m` GGUF file to Google Drive, to run on your PC with llama.cpp.

If `dataset/train.jsonl` doesn't exist yet, it does a 5-step smoke test on the 7 hand-written examples,
so you can check the notebook works before the real data is ready.

In [ ]:
%%capture
# Install (copied from Unsloth's official Qwen3.5 notebook)
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision "torchaudio==2.8.0" bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
!unsloth install-kernels
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
# Unzip local-assistant.zip into your Google Drive once, so it is at MyDrive/local-assistant.
# Everything this notebook writes goes there too, so nothing is lost if Colab disconnects.
PROJECT = "/content/drive/MyDrive/local-assistant"
%cd {PROJECT}
!ls

import sys
sys.path.insert(0, PROJECT)
TRAIN, VAL = "dataset/train.jsonl", "dataset/val.jsonl"
SMOKE_TEST = not os.path.exists(TRAIN)
if SMOKE_TEST:
    TRAIN = VAL = "dataset/seed_examples.jsonl"
    print("No train.jsonl yet: smoke test on the 7 seed examples.")

In [ ]:
from unsloth import FastVisionModel   # Qwen3.5 is a vision-language model; we only train its language part
import torch

model, processor = FastVisionModel.from_pretrained(
    "unsloth/Qwen3.5-4B",
    load_in_4bit = False,                  # 16-bit LoRA (QLoRA is not recommended for Qwen3.5)
    use_gradient_checkpointing = "unsloth",
)
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers = False,        # images are not used
    finetune_language_layers = True,
    finetune_attention_modules = True,
    finetune_mlp_modules = True,
    r = 16, lora_alpha = 16, lora_dropout = 0, bias = "none", random_state = 3407,
)
tok = getattr(processor, "tokenizer", processor)

In [ ]:
# Build the examples with the model's own chat template, in "no thinking" mode like the app
from assistant.schema import TOOLS
from train import sft_data

template = processor if getattr(processor, "chat_template", None) else tok
def apply_template(messages, add_generation_prompt):
    return template.apply_chat_template(messages, tools=TOOLS, tokenize=False,
                                        add_generation_prompt=add_generation_prompt, enable_thinking=False)

MAX_LEN = 6144
train, skipped = sft_data.build(TRAIN, tok, apply_template, MAX_LEN)
val, _ = sft_data.build(VAL, tok, apply_template, MAX_LEN)
lengths = sorted(len(e["input_ids"]) for e in train)
print(f"train {len(train)} examples, val {len(val)}, skipped {skipped}")
print(f"tokens per example: median {lengths[len(lengths)//2]}, max {lengths[-1]}")

# Look at one example: [prompt, not trained] then the part the model learns
ex = train[1]
cut = next(i for i, l in enumerate(ex["labels"]) if l != -100)
print("...", tok.decode(ex["input_ids"][cut-60:cut]), "\n==== LEARNS ====\n", tok.decode(ex["input_ids"][cut:]))

In [ ]:
from datasets import Dataset
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

bf16 = torch.cuda.is_bf16_supported()   # the T4 has no bf16, so it uses fp16
FastVisionModel.for_training(model)
trainer = Trainer(
    model = model,
    args = TrainingArguments(
        output_dir = "/content/outputs",
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        num_train_epochs = 2,
        max_steps = 5 if SMOKE_TEST else -1,
        learning_rate = 2e-4,
        warmup_ratio = 0.05,
        lr_scheduler_type = "linear",
        weight_decay = 0.001,
        optim = "adamw_8bit",
        logging_steps = 1 if SMOKE_TEST else 10,
        eval_strategy = "no" if SMOKE_TEST else "epoch",
        save_strategy = "no",
        group_by_length = True,
        bf16 = bf16, fp16 = not bf16,
        report_to = "none",
        seed = 3407,
        remove_unused_columns = False,
    ),
    train_dataset = Dataset.from_list(train),
    eval_dataset = Dataset.from_list(val),
    data_collator = DataCollatorForSeq2Seq(tok, padding = True, label_pad_token_id = -100),
)
stats = trainer.train()
print(f"{stats.metrics['train_runtime'] / 60:.1f} minutes, peak GPU memory {torch.cuda.max_memory_reserved() / 2**30:.1f} GB")

In [ ]:
# Quick look: what does the trained model answer for 3 validation prompts?
import json
FastVisionModel.for_inference(model)
records = [json.loads(l) for l in open(VAL, encoding="utf-8") if l.strip()][:3]
for rec in records:
    prompt, target = sft_data.examples(rec)[-1]
    inputs = tok(apply_template(prompt, True), return_tensors="pt", add_special_tokens=False).to("cuda")
    out = model.generate(**inputs, max_new_tokens=800, do_sample=False)
    print("USER:", prompt[1]["content"])
    print("MODEL:", tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False), "\n" + "-" * 80)

In [ ]:
# Save the adapter and a GGUF file for your PC (to Google Drive)
OUT = f"{PROJECT}/models/assistant-4b" + ("-smoketest" if SMOKE_TEST else "")
model.save_pretrained(f"{OUT}/lora")
processor.save_pretrained(f"{OUT}/lora")
model.save_pretrained_gguf("/content/gguf", processor, quantization_method = "q4_k_m")
!mkdir -p "{OUT}" && cp /content/gguf*/*.gguf "{OUT}/" && ls -lh "{OUT}"

## Run it on your PC

1. Download the `.gguf` file from `MyDrive/local-assistant/models/assistant-4b/` into `local-assistant\models\`.
   If there are two `.gguf` files, the small `mmproj` one is the image part, which this app doesn't need.
2. Get llama.cpp for Windows with **Vulkan** (works on AMD cards like the RX 5600 XT) from the
   llama.cpp releases page (`llama-*-bin-win-vulkan-x64.zip`), and start the server:
   ```
   llama-server -m models\assistant-4b.Q4_K_M.gguf --jinja -ngl 99 -c 8192 --port 8080
   ```
3. In another terminal: `set LLM_BACKEND=llama-server`, then use the app or measure it:
   ```
   python bench/evaluate.py dataset/test.jsonl --model assistant-4b
   ```
   Compare with the untuned model: `set LLM_BACKEND=ollama` and `python bench/evaluate.py dataset/test.jsonl`.

Unsloth notes that Ollama may not load Qwen3.5 GGUF files made this way, which is why this step uses llama.cpp.